# Agentic CRAG Notebook

This notebook runs directly inside the current repository. It uses the refactored `modules/` package instead of old Colab paths such as `/content/...`.

Goals:
- inspect indexed sources;
- build the retriever/vector store;
- test retrieval;
- run the QA workflow;
- optionally launch the Gradio UI.


## 1. Bootstrap project

This cell finds the project root whether the notebook is opened from the repo root or from `notebooks/`.


In [1]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "modules").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

# Lightweight notebook defaults. These avoid TensorFlow/Keras import issues.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("EMBEDDING_BACKEND", "hashing")
os.environ.setdefault("VECTOR_STORE_BACKEND", "memory")
os.environ.setdefault("LLM_TIMEOUT_SECONDS", "15")
os.environ.setdefault("LLM_MAX_RETRIES", "0")

print(f"Project root: {PROJECT_ROOT}")
print(f"Embedding backend: {os.environ['EMBEDDING_BACKEND']}")
print(f"Vector store backend: {os.environ['VECTOR_STORE_BACKEND']}")


Project root: A:\Agentic-CRAG
Embedding backend: hashing
Vector store backend: memory


## 2. Import application modules


In [2]:
from modules.ingestion.sources import list_sources, uploaded_names
from modules.rag.service import clear_retriever, get_retriever, answer_question, rebuild_store
from modules.ui.formatting import finalize_answer, format_retrieved_sources

print("Imports OK")


Imports OK


## 3. Inspect available sources


In [3]:
import pandas as pd

sources = list_sources()
pd.DataFrame(sources)


,kind,name,path,status
0,Built-in,Noi Khoa YHCT - GS Hoang Bao Chau,A:\Agentic-CRAG\Data\2010. Noi Khoa YHCT - GS ...,Available
1,Built-in,Benh Ngu Quan YHCT,A:\Agentic-CRAG\Data\bệnh ngũ quan.docx,Available
2,Built-in,Nhi Khoa YHCT,A:\Agentic-CRAG\Data\nhi-khoa-y-hoc-co-truyen....,Available
3,Built-in,noi-khoa-y-hoc-co-truyen,A:\Agentic-CRAG\Data\noi-khoa-y-hoc-co-truyen....,Available


In [4]:
print("Uploaded sources:", uploaded_names())


Uploaded sources: []


## 4. Build retriever and test retrieval

By default this notebook uses hashing embeddings, so it can run without downloading a HuggingFace model. Set `EMBEDDING_BACKEND=huggingface` before the bootstrap cell if you want to use the configured embedding model.


In [5]:
clear_retriever()
retriever = get_retriever()
print(type(retriever).__name__)


[INIT] Using offline hashing embeddings.
[PROCESS] Processing Noi Khoa YHCT - GS Hoang Bao Chau...


[INFO] Using generic fallback splitter for Noi_Khoa_YHCT_GS_Hoang_Bao_Chau.


[INFO] Noi_Khoa_YHCT_GS_Hoang_Bao_Chau: 1128 chunks
[PROCESS] Processing Benh Ngu Quan...
[INFO] Using generic fallback splitter for Benh_Ngu_Quan_YHCT.
[INFO] Benh_Ngu_Quan_YHCT: 490 chunks
[PROCESS] Processing Nhi Khoa YHCT...


[INFO] Using generic fallback splitter for Nhi_Khoa_YHCT.
[INFO] Nhi_Khoa_YHCT: 448 chunks
[PROCESS] Processing Noi Khoa General...


[INFO] noi-khoa-y-hoc-co-truyen: 1427 chunks
[PROCESS] Creating vector store with 3493 chunks...


[SUCCESS] In-memory vector store created with count: 3493


HybridRetriever


In [6]:
query = "Chứng tí là gì?"
docs = retriever.invoke(query)

rows = format_retrieved_sources(docs)
pd.DataFrame(rows, columns=["#", "Type", "Source", "Section", "Preview", "Chars"])


,#,Type,Source,Section,Preview,Chars
0,1,Local knowledge base,Noi Khoa YHCT - GS Hoang Bao Chau,ĐẠI CƯƠNG,"ĐẠI CƯƠNG: Tý là bế tắc, chứng Tý là chứng kin...",985
1,2,Local knowledge base,Noi Khoa Y Hoc Co Truyen,"DI TINH > 8. Ngũ hổ truy phong tán: Thuyền y, ...","8. Ngũ hổ truy phong tán: Thuyền y, nam tinh, ...",1745
2,3,Local knowledge base,Noi Khoa YHCT - GS Hoang Bao Chau,CHỨNG TRỊ,"CHỨNG TRỊ: Tuy chia 5 chứng tý ở cân, cốt, mạc...",187
3,4,Local knowledge base,Noi Khoa YHCT - GS Hoang Bao Chau,"1.2. Phương thuốc riêng cho các thể Hành tý, T...",ý thang (Trích từ Loại chứng trị tài - Chứng t...,994
4,5,Local knowledge base,Noi Khoa YHCT - GS Hoang Bao Chau,"5.6. Nếu huyết ứ ở lưng hông thành nuy, mạch t...","5.6. Nếu huyết ứ ở lưng hông thành nuy, mạch t...",626


In [7]:
for i, doc in enumerate(docs[:3], start=1):
    metadata = doc.metadata or {}
    preview = " ".join((doc.page_content or "").split())[:500]
    print(f"[{i}] {metadata.get('source')} - {metadata.get('section')}")
    print(preview)
    print("-" * 80)


[1] Noi_Khoa_YHCT_GS_Hoang_Bao_Chau - ĐẠI CƯƠNG:
ĐẠI CƯƠNG: Tý là bế tắc, chứng Tý là chứng kinh mạch bị ngoại tà vào làm bế tắc dẫn đến khí huyết vận hành bị trở ngại gây nên bì phu, cân cốt, cơ nhục, khớp xương đau tức ê ẩm tê bì, nặng thì khớp sưng lên co duỗi khó khăn. Hai nguyên nhân phối hợp với nhau gây nên bệnh: một là nguyên khí hư yếu, hai là phong hàn thấp ba loại tà khí thừa hư cùng xâm nhập vào kinh lạc, làm bế tắc kinh lạc, hoặc là phong hàn thấp tà uất lâu hóa nhiệt, hoặc là kinh lạc có tích nhiệt, nay lại có phong hàn thấp tà x
--------------------------------------------------------------------------------
[2] noi-khoa-y-hoc-co-truyen - DI TINH > 8. Ngũ hổ truy phong tán: Thuyền y, nam tinh, thiên ma, toàn yết,
8. Ngũ hổ truy phong tán: Thuyền y, nam tinh, thiên ma, toàn yết, cương trùng. 9. Đương quy địa hoàng thang: Đương quy, địa hoàng, bạch thược, xuyên khung, phòng phong, bạch chỉ, cảo bản, tế tân. CHỨNG TÝ (Viêm đa khớp dạng thấp) "Tý" là bế lại, là có nghĩa ngăn

## 5. Run QA workflow

This cell uses the LLM API configured in `.env`. If the API is unavailable, the cell still completes and prints a useful message.


In [8]:
question = "Chứng tí là gì?"
try:
    result, elapsed = answer_question(question)
    answer = finalize_answer(question, result.get("generation", ""), result.get("documents", []))
    print(f"Done in {elapsed:.1f}s")
    print(answer)
except Exception as exc:
    print(f"QA workflow unavailable: {exc}")
    print("Retriever cells above still work. Check .env if you want API generation.")


[INIT] Using API generation LLM provider chain: groq:llama-3.1-8b-instant, cerebras:gpt-oss-120b, mistral:mistral-small-latest, openrouter:nvidia/nemotron-nano-9b-v2:free

[PROCESS] RETRIEVAL


[PROCESS] HYBRID GRADING


[WARNING] LLM grader batch failed: Connection error.
[INFO] Re-ranker disabled. Set RERANKER_ENABLED=true to use it.
[PROCESS] GENERATE
[WARNING] Generation failed, returning retrieval fallback: LLM unavailable in this request
Done in 10.6s
LLM API đang lỗi nên hệ thống chưa tổng hợp được câu trả lời hoàn chỉnh. Dưới đây là các bằng chứng liên quan nhất tới 'Chứng tí':

- [1] Noi Khoa YHCT - GS Hoang Bao Chau - ĐẠI CƯƠNG
  ĐẠI CƯƠNG: Tý là bế tắc, chứng Tý là chứng kinh mạch bị ngoại tà vào làm bế tắc dẫn đến khí huyết vận hành bị trở ngại gây nên bì phu, cân cốt, cơ nhục, khớp xương đau tức ê ẩm tê bì, nặng thì khớp sưng lên co duỗi khó khăn. Hai nguyên nhân phối hợp với nhau gây nên bệnh: một là nguyên khí hư yếu, hai là phong hàn thấp ba loại tà khí thừa hư cùng xâm nhập vào kinh lạc, làm bế tắc kinh lạc, hoặc là phong hàn thấp tà uất lâu hóa nhiệt, hoặc là kinh lạc có tích nhiệt, nay lại có phong hàn thấp tà xâm nhập. Nội kinh chia làm 5 chứng tý: Cân tý, Cốt tý, Cơ nhục tý, Mạch t

## 6. Rebuild vector store, optional

Run this after adding or deleting files in `Data/` or `user_sources/`.


In [9]:
# Uncomment when you want to force a full rebuild.
# print(rebuild_store())


## 7. Launch Gradio, optional

This cell is commented so `Run all` does not block the kernel. Uncomment it when you want to open the UI.


In [10]:
# from modules.ui.gradio_app import launch_demo
# launch_demo()
